In [27]:
%pip install pandas

import os
from dotenv import load_dotenv
from pprint import pprint
import pandas as pd

import kalshi_python
from kalshi_python.models.get_market_response import GetMarketResponse
from kalshi_python.rest import ApiException
from kalshi_python_sync import Configuration, KalshiClient

#===================== Config ==============================
load_dotenv()
API_KEY_ID = os.getenv("KALSHI_KEY_ID")
PRIVATE_KEY_PATH = "kalshi.pem"

def configure_client(path: str):
    config = Configuration(
        host="https://api.elections.kalshi.com/trade-api/v2"
    )

    with open(path, "r") as f:
        config.private_key_pem = f.read()

    config.api_key_id = API_KEY_ID

    client = KalshiClient(config)

    balance = client.get_balance()
    print(f"Balance: ${balance.balance / 100:.2f}")

    return client, balance



  Using cached numpy-2.4.2-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 11.6 MB/s eta 0:00:0000:0100:01
Using cached numpy-2.4.2-cp313-cp313-macosx_14_0_arm64.whl (5.2 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
def fetch_all_markets(client):
    all_markets = []
    cursor = None

    for _ in range(10):
        response = client.get_markets(limit=1000, cursor=cursor)

        markets = response.markets
        all_markets.extend(markets)

        cursor = response.cursor

        if cursor is None:
            break

    print(f"Fetched {len(all_markets)} markets")
    return all_markets
    
def markets_to_rows(markets):
    rows = []

    for m in markets:
        rows.append({
            "ticker": m.ticker,
            "title": m.title,
            "status": m.status,
            "yes_bid": m.yes_bid,
            "yes_ask": m.yes_ask,
            "last_price": m.last_price,
            "volume": m.volume,
            "open_interest": m.open_interest,
            "close_time": m.close_time,
            "event_ticker": m.event_ticker,
        })

    return rows

def save_markets_csv(rows, filename="kalshi_markets.csv"):
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} rows to {filename}")

In [30]:
client, balance = configure_client(PRIVATE_KEY_PATH)

markets = fetch_all_markets(client)

rows = markets_to_rows(markets)

save_markets_csv(rows)

Balance: $49.13


KeyboardInterrupt: 